# Example: Reading and writing an equilibrium to IMAS (via IMAS-Python)

In this example, we take a MAST-U-like equilibrium and show how to save it to, and load it back from, the [IMAS](https://imas.iter.org/) `equilibrium` IDS using the [IMAS-Python](https://github.com/iterorganization/IMAS-Python) package. IDSs (Interface Data Structures) are the standardised data structures IMAS uses to link different modelling codes together; here we serialise the IDS to a netCDF file.

As in the previous example, we run our standard equilibrium solve - making it lower single null by pushing the P6 current.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# initialise equilibrium object
from freegsnke import equilibrium_update
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

# initialise profile object
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8.1e3,
    Ip=6.2e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2
)

# initialise solver
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)

# set coil currents
import pickle
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak[key].current = current_values[key]

# change this to shift plasma vertically up (-) or down (+)
eq.tokamak["P6"].current += 500

# carry out forward solve
GSStaticSolver.solve(eq=eq,
                     profiles=profiles,
                     constrain=None,
                     target_relative_tolerance=1e-9)

# plot the resulting equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

## Writing to the equilibrium IDS

FreeGSNKE provides a helper module, `freegsnke.imas_read_write`, that fills in the relevant fields of an IMAS `equilibrium` IDS from a solved `eq`/`profiles` pair: high-level `ids_properties`/`code` metadata, `vacuum_toroidal_field`, and a single `time_slice` containing the plasma boundary, global quantities, 1D flux-surface profiles, and three 2D poloidal flux maps on FreeGSNKE's rectangular (R, Z) grid - `total` (also carrying `j_phi`), `plasma`-only, and `vacuum` (coil-only) - tagged using the standard `equilibrium_profiles_2d_identifier` codes.

See the [equilibrium IDS data dictionary](https://imas-data-dictionary.readthedocs.io/en/latest/generated/ids/equilibrium.html) for the full structure.

In [ ]:
from freegsnke import imas_read_write

equilibrium_ids = imas_read_write.write_equilibrium_to_ids(eq, profiles)

Now that it has been built, we can inspect the overall structure using IMAS-Python's own utilities.

In [ ]:
import imas

# inspect at a high level
imas.util.inspect(equilibrium_ids, hide_empty_nodes=True)

We can inspect at lower levels of the structure too.

In [ ]:
imas.util.inspect(equilibrium_ids.time_slice[0].global_quantities, hide_empty_nodes=True)

If unsure of where exactly a quantity is located in the structure, you can search for certain strings. Note that this returns all possible locations in the data dictionary, without revealing whether that location is actually filled.

In [ ]:
imas.util.find_paths(equilibrium_ids, "psi_boundary")

## Writing to netCDF

IMAS-Python IDSs are written to disk via an `imas.DBEntry`. `imas_read_write.save_equilibrium_ids` wraps this for the netCDF backend - see [here](https://imas-python.readthedocs.io/en/stable/netcdf.html) for more details.

In [ ]:
imas_read_write.save_equilibrium_ids(equilibrium_ids, "MASTU_LSN_equilibrium.nc")

## Reading from netCDF

Now we can read the equilibrium IDS back from the netCDF file.

An `equilibrium` IDS doesn't store individual coil currents (only the boundary, global quantities, and profile data), so - just as with the geqdsk example - we use the inverse solver to re-build the equilibrium from what's actually saved.

We build the constraints entirely from what the IDS gives us:

- an **isoflux constraint** from the plasma boundary outline (`time_slice.boundary.outline`) - all these points should sit on the same flux contour, without us needing to know its absolute value;
- **null-point constraints** at the magnetic axis (`time_slice.global_quantities.magnetic_axis`);
- a **`psi_vals` constraint built from the *entire* saved 2D `psi` map** (`time_slice.profiles_2d[0].psi`) - unlike the constraints above, this one is given the actual flux *values*, not just which points should share a value, so it's what lets the optimiser pin down the absolute flux level (and, with it, the individual coil currents) rather than just the plasma shape. Passing the grid's own R/Z axes (rather than a flattened list of points) triggers `Inverse_optimizer`'s "full grid" fast path, which reuses the equilibrium's own cached Greens functions instead of recomputing them pointwise.

To check that the IDS faithfully captures the equilibrium, we'll reconstruct it purely from what's in the file: the grid extent, the plasma current, and the `dpressure_dpsi`/`f_df_dpsi` profile data.

FreeGSNKE's `GeneralPprimeFFprime` profile class accepts exactly this kind of tabulated profile data directly, rather than a parametric form like `ConstrainPaxisIp` above.

We reset the coil currents on a **fresh, independent** tokamak to zero first, to make sure we aren't just reusing the answer.

In [ ]:
loaded_ids = imas_read_write.load_equilibrium_ids("MASTU_LSN_equilibrium.nc")
time_slice = loaded_ids.time_slice[0]

In [ ]:
# grid data (IDS stores psi in Wb, i.e. with the 2*pi factor included - FreeGSNKE works in Wb/rad)
R_read = np.array(time_slice.profiles_2d[0].grid.dim1)
Z_read = np.array(time_slice.profiles_2d[0].grid.dim2)
psi_read = np.array(time_slice.profiles_2d[0].psi) / (2 * np.pi)

Rmin, Rmax = R_read[0], R_read[-1]
Zmin, Zmax = Z_read[0], Z_read[-1]
nx_read, ny_read = len(R_read), len(Z_read)

# profile data (the IDS stores dp/dpsi and FF' with respect to psi in Wb, and the
# reader converts them back to FreeGSNKE's Wb/rad convention)
profile_data = imas_read_write.read_profiles_from_equilibrium_ids(loaded_ids)
psi_n_read = profile_data["psi_n"]
pprime_read = profile_data["pprime"]
ffprime_read = profile_data["ffprime"]
Ip_read = profile_data["Ip"]
fvac_read = profile_data["fvac"]

# boundary and magnetic axis
rbdry, zbdry = np.array(time_slice.boundary.outline.r), np.array(time_slice.boundary.outline.z)
rmagx, zmagx = time_slice.global_quantities.magnetic_axis.r, time_slice.global_quantities.magnetic_axis.z

Now let's set up and run the inverse solver, using those quantities plus the constraints described above to reconstruct the coil currents.

In [ ]:
from freegsnke.jtor_update import GeneralPprimeFFprime
from freegsnke.inverse import Inverse_optimizer

# coil names
active_coil_names = tokamak.coils_list[: tokamak.n_active_coils]

# fresh tokamak so we can set coil currents to zero and compare to old eq object
tokamak_inv = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# fresh equilibrium object
eq_inv = equilibrium_update.Equilibrium(
    tokamak=tokamak_inv,
    Rmin=Rmin, Rmax=Rmax,
    Zmin=Zmin, Zmax=Zmax,
    nx=nx_read,
    ny=ny_read,
)

# fresh profiles object, built from the tabulated data saved to the IDS
profiles_inv = GeneralPprimeFFprime(
    eq=eq_inv,
    Ip=Ip_read,
    fvac=fvac_read,
    psi_n=psi_n_read,
    pprime_data=pprime_read,
    ffprime_data=ffprime_read,
)

# isoflux constraint: subsample the boundary outline stored in the IDS
isoflux_set = [[rbdry[::5], zbdry[::5]]]

# null-point constraint: magnetic axis (from the IDS)
null_points = [
    [rmagx],
    [zmagx],
]

# psi_vals constraint built from the *entire* saved 2D psi map (full-grid fast path)
psi_vals = [eq_inv.R, eq_inv.Z, psi_read]

# build constraints
constrain = Inverse_optimizer(
    isoflux_set=isoflux_set,
    null_points=null_points,
    psi_vals=psi_vals,
)

# initialise solver object
GSStaticSolver_inv = GSstaticsolver.NKGSsolver(eq_inv)

In [ ]:
# set coils to zero current
for label in active_coil_names:
    eq_inv.tokamak.set_coil_current(label, 0.0)

# default plasma flux guess
eq_inv.plasma_psi = eq_inv.create_psi_plasma_default()

# solve
GSStaticSolver_inv.solve(
    eq=eq_inv,
    profiles=profiles_inv,
    constrain=constrain,
    target_relative_tolerance=1e-4,
    target_relative_psit_update=1e-3,
    max_solving_iterations=50,
    l2_reg=1e-9,
    verbose=True,
    Picard_handover=1e-5,
    full_jacobian_handover=[2e-2,1e-2],
)

This converges properly and gives the correct total flux, coil currents close to their true values, and `psi_axis`/`psi_bndry` matching to several decimal places.

In [ ]:
# print coil currents
reconstructed_currents = {label: eq_inv.tokamak[label].current for label in active_coil_names}
original_active_currents = {label: eq.tokamak[label].current for label in active_coil_names}

print("Coil currents [A]:")
print(f"{'coil':>10s}  {'original':>12s}  {'reconstructed':>14s}")
for label in active_coil_names:
    print(f"{label:>10s}  {original_active_currents[label]:12.1f}  {reconstructed_currents[label]:14.1f}")

# some small differences
psi_diff_inv = np.abs(eq.psi() - eq_inv.psi())
print(f"\nMagnetic axis:   original = {eq.magneticAxis()[0:2]}, reconstructed = {eq_inv.magneticAxis()[0:2]}")
print(f"psi_axis:        original = {eq.psi_axis:.6g}, reconstructed = {eq_inv.psi_axis:.6g}")
print(f"psi_boundary:    original = {eq.psi_bndry:.6g}, reconstructed = {eq_inv.psi_bndry:.6g}")
print(f"Max |Δpsi| relative to max|psi|: {np.max(psi_diff_inv) / np.max(np.abs(eq.psi())):.3e}")

# plot the two equilibria side by side
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 8), dpi=80)
for ax, e, title in [(ax1, eq, "Original"), (ax2, eq_inv, "Reconstruction from IDS")]:
    ax.grid(True, which="both")
    e.plot(axis=ax, show=False)
    e.tokamak.plot(axis=ax, show=False)
    ax.set_xlim(0.1, 2.15)
    ax.set_ylim(-2.25, 2.25)
    ax.set_title(title)
    if ax == ax2:
        constrain.plot(axis=ax2)
plt.tight_layout()

In the future, we'll work on adding in capabilities to save the machine description and coil currents to the `pf_active` and `pf_passive` IDS formats. 